In [5]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import plotly.io as pio
import pandas as pd
import wandb
from tqdm import tqdm
from torch.utils.data import DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

from model import Net
from train import train_pipeline, val_pipeline
from datasets import CubeObstacle, CylinderObstacle, TrainDataset, BlockageDataset
from utils.tools import calc_loss, calc_sig_strength, calc_sig_strength_gpu, probabilistic_channel_model
from utils.config import Hyperparameters as hp

random_seed = 42
batch_size = 1024
epochs = 10000   
lr = 5e-5

In [6]:
# ls models
dir_path = './models/train_model'

files_ls = os.listdir(dir_path)
files_ls = [file for file in files_ls if file.endswith('.pt')]
model_epoch = [int(file.split('_')[-1].split('.')[0]) for file in files_ls]
model_dict = dict(zip(model_epoch, files_ls))
model_dict = sorted(model_dict)

In [7]:
# define the obstacles

# Create obstacles and convert to torch tensors

torch.manual_seed(random_seed)
np.random.seed(random_seed)
if hp.device == "cuda":
    torch.cuda.manual_seed_all(random_seed)

obstacle_ls = [
    CubeObstacle(-30, 25, 35, 60, 20, 0.1),
    CubeObstacle(-30, -25, 45, 10, 35, 0.1),
    CubeObstacle(-30, -60, 35, 60, 20, 0.1),
    CubeObstacle(50, -20, 35, 25, 25, 0.1),
    CylinderObstacle(10, -5,  70, 15, 0.1),
]

obst_points = []
for obstacle in obstacle_ls:
    obst_points.append(torch.tensor(obstacle.points, dtype=torch.float32))

obst_points = torch.cat([op for op in obst_points], dim=1).mT.to(hp.device)

### Base line model definition

1. Zero coordinates $(0, 0, \mathbf{x}_z)$
2. Centroid of the coordinates(Average of the coordinates)
    $$\frac{1}{N}\sum_{k \in K}\mathbf{u}_k + \begin{bmatrix}0\\ 0\\ \mathbf{x}_z\end{bmatrix}$$
3. Probabilistic channel model
4. Blockage channel model (Brute force)

In [ ]:
gn_num_ls = [2, 3, 4, 5, 6, 7, 8]

for gn_num in gn_num_ls:
    torch.manual_seed(random_seed)
    np.random.seed(random_seed)
    if hp.device == "cuda":
        torch.cuda.manual_seed_all(random_seed)

    wandb.init(project="DL-based UAV Positioning", name=f"train model_gn{gn_num}", config={
        "batch_size": batch_size,
        "epochs": epochs,
        "random_seed": random_seed,
        "learning_rate": lr,
        "gn_num": gn_num
    })

    dataset = BlockageDataset(100000, obstacle_ls, gn_num, dtype=torch.float32)
    x = dataset.gnd_nodes[:, :, :2].reshape(-1, 2*gn_num)
    scaler_x = MinMaxScaler(feature_range=(0, 1))
    scaler_x.fit(np.ones((2, x.shape[1]), dtype=np.float32)*np.array([[-100, 100]]).mT)
    x_scaled = scaler_x.transform(x)
    x_train, x_val = train_test_split(x_scaled, test_size=0.2, random_state=random_seed)

    train_dataset = TrainDataset(x_train, dtype=torch.float32).to(hp.device)
    val_dataset = TrainDataset(x_val, dtype=torch.float32).to(hp.device)

    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = Net(x_train.shape[1], 1024, 4, output_N=2).to(hp.device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_loss = float('inf')
    best_epoch = 0
    gn_coords = []
    for epoch in range(epochs):
        model.train()
        train_loss = train_pipeline(model, train_dataloader, optimizer, scaler_x, obst_points, hp.device, gn_num=gn_num)
        visual = False
        if epoch % 500 == 0 or epoch == epochs-1: visual=True
        val_result = val_pipeline(model, val_dataloader, scaler_x, obst_points, hp.device, visual=visual, current_epoch=epoch, obstacle_ls=obstacle_ls, gn_num=gn_num)
        val_loss = val_result['val_loss']

        train_loss /= len(train_dataloader)
        val_loss /= len(val_dataloader)

        if val_loss < best_loss:
            best_loss = val_loss
            torch.save(model.state_dict(), f'./models/gn_num_test/best_gn_num_{gn_num}.pt')

        if epoch % 500 == 0 or epoch == epochs - 1:
            print(f"Epoch: {epoch}, Train Loss: {train_loss}, Validation Loss: {val_loss}")
        if epoch == epochs - 1:
            gn_coords = val_result['gn_coords']
            gn_coords = [coord.reshape(-1, gn_num*3) for coord in gn_coords]
            gn_coords = np.concatenate(gn_coords, axis=0)
        wandb.log({
            f"train_loss": train_loss,
            f"val_loss": val_loss,
            "epoch": epoch + 1
        })

    pd.DataFrame(gn_coords).to_csv(f'./data/gn_coords_{gn_num}.csv', index=False, header=False)
    print(f"Best loss: {best_loss} at epoch {best_epoch}")
    os.rename(f'./models/gn_num_test/best_gn_num_{gn_num}.pt',
              f'./models/gn_num_test/best_gn_num_{gn_num}_epoch_{best_epoch}.pt')
    torch.save(model.state_dict(), f'./models/gn_num_test/gn_num_{gn_num}_epoch_{epochs-1}.pt')
    wandb.finish()

Validation: 100%|██████████| 20/20 [00:00<00:00, 389.09it/s]


Epoch: 0, Train Loss: -11.80221092851856, Validation Loss: -12.053387546539307


Validation: 100%|██████████| 20/20 [00:00<00:00, 304.85it/s]


Epoch: 500, Train Loss: -12.456857476053358, Validation Loss: -12.497464323043824


Validation: 100%|██████████| 20/20 [00:00<00:00, 339.07it/s]


...

Validation: 100%|██████████| 20/20 [00:00<00:00, 200.31it/s]


Epoch: 6500, Train Loss: -12.069326618049718, Validation Loss: -12.06752119064331


Validation: 100%|██████████| 20/20 [00:00<00:00, 199.19it/s]


Epoch: 7000, Train Loss: -12.069655358036862, Validation Loss: -12.067240810394287


Training:  43%|████▎     | 34/79 [00:00<00:00, 65.77it/s]